# Auditoria: bases calculadas (D/F) e saídas ARIMA (`temporal_fusion`)

Verifica:
- Existência de parquets, linhas/colunas, schema
- `% NaN` / `inf` na precipitação e colunas `tsf_arima_*`
- Duplicatas em `(cidade_norm, ts_hour)`
- Amostra de estatísticas e comparação **calculado vs ARIMA** no mesmo ano

**Execução:** abra na raiz do projeto (`TCC/`) ou ajuste `PROJECT_ROOT` na célula seguinte.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd
import yaml

try:
    from IPython.display import display
except ImportError:

    def display(obj):
        print(obj.to_string() if hasattr(obj, "to_string") else obj)


try:
    import pyarrow.parquet as pq
except ImportError:
    pq = None

# Raiz do repositório (pasta que contém config.yaml)
_here = Path.cwd().resolve()
PROJECT_ROOT = _here
if not (PROJECT_ROOT / "config.yaml").exists():
    for p in [_here, *_here.parents]:
        if (p / "config.yaml").exists():
            PROJECT_ROOT = p
            break

if not (PROJECT_ROOT / "config.yaml").exists():
    raise FileNotFoundError(
        "config.yaml não encontrado. Rode o kernel com cwd = raiz do projeto TCC "
        "ou defina PROJECT_ROOT manualmente."
    )

with (PROJECT_ROOT / "config.yaml").open("r", encoding="utf-8") as f:
    CFG = yaml.safe_load(f)

def _expand(p: str) -> Path:
    path = Path(p)
    return path if path.is_absolute() else (PROJECT_ROOT / path).resolve()

MODELING = _expand(CFG["paths"]["data"]["modeling"])
TEMPORAL_FUSION = _expand(CFG["paths"]["data"]["temporal_fusion"])

DIR_D_CALC = MODELING / "base_D_with_rad_drop_rows_calculated"
DIR_F_CALC = MODELING / "base_F_full_original_calculated"
DIR_D_ARIMA = TEMPORAL_FUSION / "base_D_with_rad_drop_rows_calculated" / "arima"
DIR_F_ARIMA = TEMPORAL_FUSION / "base_F_full_original_calculated" / "arima"

COL_PRECIP = "PRECIPITAÇÃO TOTAL, HORÁRIO (mm)"
TSF_PRED = "tsf_arima_precip_pred"
TSF_RESID = "tsf_arima_precip_resid"
KEYS = ["cidade_norm", "ts_hour"]

print("PROJECT_ROOT:", PROJECT_ROOT)
print("MODELING:", MODELING)
print("TEMPORAL_FUSION:", TEMPORAL_FUSION)

PROJECT_ROOT: D:\Projetos\TCC
MODELING: D:\Projetos\TCC\data\modeling
TEMPORAL_FUSION: D:\Projetos\TCC\data\temporal_fusion


## Funções de auditoria

In [2]:
def list_year_parquets(folder: Path) -> List[Path]:
    if not folder.is_dir():
        return []
    return sorted(folder.glob("inmet_bdq_*_cerrado.parquet"))


def parquet_meta(path: Path) -> Dict[str, Any]:
    out: Dict[str, Any] = {"file": path.name, "exists": path.exists()}
    if not path.exists() or pq is None:
        return out
    pf = pq.ParquetFile(path)
    out["n_rows"] = pf.metadata.num_rows
    out["n_cols"] = len(pf.schema_arrow.names)
    out["columns"] = list(pf.schema_arrow.names)
    return out


def audit_precip_and_keys(
    path: Path,
    check_tsf: bool = False,
) -> Dict[str, Any]:
    """Lê só colunas necessárias para não estourar RAM desnecessariamente."""
    row: Dict[str, Any] = {"file": path.name}
    if not path.exists():
        row["error"] = "missing"
        return row

    meta = parquet_meta(path)
    all_cols = meta.get("columns", [])
    cols = [c for c in KEYS + [COL_PRECIP] if c in all_cols]
    for c in (TSF_PRED, TSF_RESID):
        if check_tsf and c in all_cols and c not in cols:
            cols.append(c)

    try:
        df = pd.read_parquet(path, columns=cols)
    except Exception as e:
        row["error"] = str(e)
        return row

    row["n_rows"] = len(df)
    if COL_PRECIP in df.columns:
        s = pd.to_numeric(df[COL_PRECIP], errors="coerce")
        row["precip_nan_pct"] = float(s.isna().mean() * 100)
        arr = s.to_numpy(dtype=float, copy=False)
        row["precip_inf_pct"] = float(np.isinf(arr).mean() * 100) if len(arr) else 0.0
        finite = s[np.isfinite(s)]
        row["precip_min"] = float(finite.min()) if len(finite) else np.nan
        row["precip_max"] = float(finite.max()) if len(finite) else np.nan
        row["precip_mean"] = float(finite.mean()) if len(finite) else np.nan
        row["precip_zero_pct"] = float((finite == 0).mean() * 100) if len(finite) else np.nan
    else:
        row["precip_nan_pct"] = np.nan
        row["note"] = f"coluna '{COL_PRECIP}' ausente"

    if all(k in df.columns for k in KEYS):
        dup = df.duplicated(subset=KEYS, keep=False)
        row["dup_keys_pct"] = float(dup.mean() * 100)
        row["dup_keys_n"] = int(dup.sum())
    else:
        row["dup_keys_pct"] = np.nan
        row["missing_keys"] = [k for k in KEYS if k not in df.columns]

    if check_tsf:
        for c in (TSF_PRED, TSF_RESID):
            if c in df.columns:
                v = pd.to_numeric(df[c], errors="coerce")
                row[f"{c}_nan_pct"] = float(v.isna().mean() * 100)
                row[f"{c}_finite_pct"] = float(np.isfinite(v).mean() * 100)
            else:
                row[f"{c}_nan_pct"] = np.nan

    return row


def audit_folder(label: str, folder: Path, check_tsf: bool = False) -> pd.DataFrame:
    files = list_year_parquets(folder)
    if not files:
        print(f"[{label}] Pasta vazia ou inexistente: {folder}")
        return pd.DataFrame()
    rows = [audit_precip_and_keys(p, check_tsf=check_tsf) for p in files]
    df = pd.DataFrame(rows)
    df.insert(0, "base", label)
    return df

## 1) Inventário rápido (schema + num_rows via PyArrow, sem carregar tudo)

In [3]:
def inventory_table(folder: Path, label: str) -> pd.DataFrame:
    rows = []
    for p in list_year_parquets(folder):
        m = parquet_meta(p)
        rows.append({
            "base": label,
            "file": m["file"],
            "n_rows": m.get("n_rows"),
            "n_cols": m.get("n_cols"),
            "has_precip": COL_PRECIP in (m.get("columns") or []),
            "has_tsf_arima": TSF_PRED in (m.get("columns") or []),
        })
    return pd.DataFrame(rows)

for name, d in [
    ("D_calculated", DIR_D_CALC),
    ("F_calculated", DIR_F_CALC),
    ("D_arima", DIR_D_ARIMA),
    ("F_arima", DIR_F_ARIMA),
]:
    t = inventory_table(d, name)
    print(f"\n=== {name} ({d}) — {len(t)} arquivos ===")
    display(t)


=== D_calculated (D:\Projetos\TCC\data\modeling\base_D_with_rad_drop_rows_calculated) — 22 arquivos ===


,base,file,n_rows,n_cols,has_precip,has_tsf_arima
0,D_calculated,inmet_bdq_2003_cerrado.parquet,151024,27,True,False
1,D_calculated,inmet_bdq_2004_cerrado.parquet,133224,27,True,False
2,D_calculated,inmet_bdq_2005_cerrado.parquet,231484,27,True,False
3,D_calculated,inmet_bdq_2006_cerrado.parquet,310984,27,True,False
4,D_calculated,inmet_bdq_2007_cerrado.parquet,1263964,27,True,False
5,D_calculated,inmet_bdq_2008_cerrado.parquet,2244832,27,True,False
6,D_calculated,inmet_bdq_2009_cerrado.parquet,2465520,27,True,False
7,D_calculated,inmet_bdq_2010_cerrado.parquet,2455408,27,True,False
8,D_calculated,inmet_bdq_2011_cerrado.parquet,2410808,27,True,False
9,D_calculated,inmet_bdq_2012_cerrado.parquet,2492060,27,True,False



=== F_calculated (D:\Projetos\TCC\data\modeling\base_F_full_original_calculated) — 22 arquivos ===


,base,file,n_rows,n_cols,has_precip,has_tsf_arima
0,F_calculated,inmet_bdq_2003_cerrado.parquet,396056,27,True,False
1,F_calculated,inmet_bdq_2004_cerrado.parquet,396488,27,True,False
2,F_calculated,inmet_bdq_2005_cerrado.parquet,597904,27,True,False
3,F_calculated,inmet_bdq_2006_cerrado.parquet,604260,27,True,False
4,F_calculated,inmet_bdq_2007_cerrado.parquet,2363172,27,True,False
5,F_calculated,inmet_bdq_2008_cerrado.parquet,4000732,27,True,False
6,F_calculated,inmet_bdq_2009_cerrado.parquet,4371032,27,True,False
7,F_calculated,inmet_bdq_2010_cerrado.parquet,4423228,27,True,False
8,F_calculated,inmet_bdq_2011_cerrado.parquet,4596208,27,True,False
9,F_calculated,inmet_bdq_2012_cerrado.parquet,4885352,27,True,False



=== D_arima (D:\Projetos\TCC\data\temporal_fusion\base_D_with_rad_drop_rows_calculated\arima) — 22 arquivos ===


,base,file,n_rows,n_cols,has_precip,has_tsf_arima
0,D_arima,inmet_bdq_2003_cerrado.parquet,151024,29,True,True
1,D_arima,inmet_bdq_2004_cerrado.parquet,133224,29,True,True
2,D_arima,inmet_bdq_2005_cerrado.parquet,231484,29,True,True
3,D_arima,inmet_bdq_2006_cerrado.parquet,310984,29,True,True
4,D_arima,inmet_bdq_2007_cerrado.parquet,1263964,29,True,True
5,D_arima,inmet_bdq_2008_cerrado.parquet,2244832,29,True,True
6,D_arima,inmet_bdq_2009_cerrado.parquet,2465520,29,True,True
7,D_arima,inmet_bdq_2010_cerrado.parquet,2455408,29,True,True
8,D_arima,inmet_bdq_2011_cerrado.parquet,2410808,29,True,True
9,D_arima,inmet_bdq_2012_cerrado.parquet,2492060,29,True,True



=== F_arima (D:\Projetos\TCC\data\temporal_fusion\base_F_full_original_calculated\arima) — 22 arquivos ===


,base,file,n_rows,n_cols,has_precip,has_tsf_arima
0,F_arima,inmet_bdq_2003_cerrado.parquet,396056,29,True,True
1,F_arima,inmet_bdq_2004_cerrado.parquet,396488,29,True,True
2,F_arima,inmet_bdq_2005_cerrado.parquet,597904,29,True,True
3,F_arima,inmet_bdq_2006_cerrado.parquet,604260,29,True,True
4,F_arima,inmet_bdq_2007_cerrado.parquet,2363172,29,True,True
5,F_arima,inmet_bdq_2008_cerrado.parquet,4000732,29,True,True
6,F_arima,inmet_bdq_2009_cerrado.parquet,4371032,29,True,True
7,F_arima,inmet_bdq_2010_cerrado.parquet,4423228,29,True,True
8,F_arima,inmet_bdq_2011_cerrado.parquet,4596208,29,True,True
9,F_arima,inmet_bdq_2012_cerrado.parquet,4885352,29,True,True


## 2) Auditoria pesada: precip, duplicatas de chave, e (se existir) `tsf_arima_*`

Pode demorar em bases F (milhões de linhas por ano). Se precisar, restrinja anos na célula abaixo.

In [4]:
# None = todos os parquets encontrados; ou ex.: {2010, 2011}
YEARS_FILTER: Optional[set] = None


def filter_paths(paths: List[Path]) -> List[Path]:
    if not YEARS_FILTER:
        return paths
    out = []
    for p in paths:
        try:
            y = int(p.stem.split("_")[2])
        except Exception:
            continue
        if y in YEARS_FILTER:
            out.append(p)
    return out


def run_audits():
    results = []
    specs = [
        ("D_calculated", DIR_D_CALC, False),
        ("F_calculated", DIR_F_CALC, False),
        ("D_arima", DIR_D_ARIMA, True),
        ("F_arima", DIR_F_ARIMA, True),
    ]
    for label, folder, tsf in specs:
        files = filter_paths(list_year_parquets(folder))
        if not files:
            continue
        for p in files:
            r = audit_precip_and_keys(p, check_tsf=tsf)
            r["base"] = label
            results.append(r)
    return pd.DataFrame(results)

df_audit = run_audits()
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
df_audit

,file,n_rows,precip_nan_pct,precip_inf_pct,precip_min,precip_max,precip_mean,precip_zero_pct,dup_keys_pct,dup_keys_n,base,tsf_arima_precip_pred_nan_pct,tsf_arima_precip_pred_finite_pct,tsf_arima_precip_resid_nan_pct,tsf_arima_precip_resid_finite_pct
0,inmet_bdq_2003_cerrado.parquet,151024,0.000000,0.0,0.0,66.800000,0.221808,93.246107,100.0,151024,D_calculated,NaN,NaN,NaN,NaN
1,inmet_bdq_2004_cerrado.parquet,133224,0.000000,0.0,0.0,58.600000,0.264805,91.289858,100.0,133224,D_calculated,NaN,NaN,NaN,NaN
2,inmet_bdq_2005_cerrado.parquet,231484,0.000000,0.0,0.0,72.000000,0.219105,92.777039,100.0,231484,D_calculated,NaN,NaN,NaN,NaN
3,inmet_bdq_2006_cerrado.parquet,310984,0.000000,0.0,0.0,60.000000,0.219229,92.726314,100.0,310984,D_calculated,NaN,NaN,NaN,NaN
4,inmet_bdq_2007_cerrado.parquet,1263964,0.000000,0.0,0.0,86.400000,0.144599,94.655544,100.0,1263964,D_calculated,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83,inmet_bdq_2020_cerrado.parquet,6885872,33.126785,0.0,0.0,73.000000,0.030841,99.320796,100.0,6885872,F_arima,100.0,0.0,100.0,0.0
84,inmet_bdq_2021_cerrado.parquet,6937128,54.026104,0.0,0.0,61.000000,0.031472,99.285479,100.0,6937128,F_arima,100.0,0.0,100.0,0.0
85,inmet_bdq_2022_cerrado.parquet,6376552,32.881156,0.0,0.0,86.599998,0.143666,94.018320,100.0,6376552,F_arima,100.0,0.0,100.0,0.0
86,inmet_bdq_2023_cerrado.parquet,3888996,23.881639,0.0,0.0,87.000000,0.135584,94.976083,100.0,3888996,F_arima,100.0,0.0,100.0,0.0


In [5]:
print(df_audit.to_string())

                              file   n_rows  precip_nan_pct  precip_inf_pct  precip_min  precip_max  precip_mean  precip_zero_pct  dup_keys_pct  dup_keys_n          base  tsf_arima_precip_pred_nan_pct  tsf_arima_precip_pred_finite_pct  tsf_arima_precip_resid_nan_pct  tsf_arima_precip_resid_finite_pct
0   inmet_bdq_2003_cerrado.parquet   151024        0.000000             0.0         0.0   66.800000     0.221808        93.246107         100.0      151024  D_calculated                            NaN                               NaN                             NaN                                NaN
1   inmet_bdq_2004_cerrado.parquet   133224        0.000000             0.0         0.0   58.600000     0.264805        91.289858         100.0      133224  D_calculated                            NaN                               NaN                             NaN                                NaN
2   inmet_bdq_2005_cerrado.parquet   231484        0.000000             0.0         0.0   72.0

## 3) Resumo: onde o ARIMA "não preencheu" (`tsf_*` só NaN?)

In [6]:
if df_audit is None or df_audit.empty:
    print("df_audit vazio — rode a célula de auditoria pesada antes.")
elif "base" not in df_audit.columns:
    print("df_audit sem coluna 'base'.")
else:
    sub = df_audit[df_audit["base"].str.endswith("_arima")].copy()
    if sub.empty:
        print("Nenhuma linha *_arima em df_audit (gere parquets em temporal_fusion/.../arima).")
    else:
        cols_show = [
            "base", "file",
            "n_rows",
            "precip_nan_pct",
            "dup_keys_pct",
            "tsf_arima_precip_pred_nan_pct",
            "tsf_arima_precip_pred_finite_pct",
        ]
        cols_show = [c for c in cols_show if c in sub.columns]
        display(sub[cols_show])

,base,file,n_rows,precip_nan_pct,dup_keys_pct,tsf_arima_precip_pred_nan_pct,tsf_arima_precip_pred_finite_pct
44,D_arima,inmet_bdq_2003_cerrado.parquet,151024,0.000000,100.0,100.0,0.0
45,D_arima,inmet_bdq_2004_cerrado.parquet,133224,0.000000,100.0,100.0,0.0
46,D_arima,inmet_bdq_2005_cerrado.parquet,231484,0.000000,100.0,100.0,0.0
47,D_arima,inmet_bdq_2006_cerrado.parquet,310984,0.000000,100.0,100.0,0.0
48,D_arima,inmet_bdq_2007_cerrado.parquet,1263964,0.000000,100.0,100.0,0.0
49,D_arima,inmet_bdq_2008_cerrado.parquet,2244832,0.000000,100.0,100.0,0.0
50,D_arima,inmet_bdq_2009_cerrado.parquet,2465520,0.000000,100.0,100.0,0.0
51,D_arima,inmet_bdq_2010_cerrado.parquet,2455408,0.000000,100.0,100.0,0.0
52,D_arima,inmet_bdq_2011_cerrado.parquet,2410808,0.000000,100.0,100.0,0.0
53,D_arima,inmet_bdq_2012_cerrado.parquet,2492060,0.000000,100.0,100.0,0.0


## 4) Comparação par a par (mesmo ano): calculado vs ARIMA

Alinha por `(cidade_norm, ts_hour)` e verifica contagens e % NaN nas colunas ARIMA após o merge.

In [7]:
COMPARE_YEAR = 2010  # altere se não existir esse ano


def compare_calc_vs_arima(calc_dir: Path, arima_dir: Path, year: int, label: str):
    fn = f"inmet_bdq_{year}_cerrado.parquet"
    pc, pa = calc_dir / fn, arima_dir / fn
    if not pc.exists():
        print(f"[{label}] Falta calculado: {pc}")
        return
    if not pa.exists():
        print(f"[{label}] Falta arima: {pa}")
        return

    cols = [c for c in KEYS + [COL_PRECIP] if c in parquet_meta(pc).get("columns", [])]
    dfc = pd.read_parquet(pc, columns=cols)
    dfa = pd.read_parquet(pa, columns=cols + [TSF_PRED, TSF_RESID])

    print(f"\n===== {label} | {year} =====")
    print(f"Linhas calculado: {len(dfc):,} | Linhas arima: {len(dfa):,}")

    merged = dfc.merge(
        dfa[KEYS + [TSF_PRED, TSF_RESID]],
        on=KEYS,
        how="outer",
        indicator=True,
    )
    vc = merged["_merge"].value_counts()
    print("Merge outer:", vc.to_dict())

    if TSF_PRED in merged.columns:
        pred = pd.to_numeric(merged[TSF_PRED], errors="coerce")
        print(f"tsf_arima_precip_pred — NaN: {pred.isna().mean()*100:.2f}% | finitos: {np.isfinite(pred).mean()*100:.2f}%")

    # Uma cidade com mais linhas (amostra)
    if "cidade_norm" in dfc.columns:
        top = dfc["cidade_norm"].value_counts().index[0]
        s = dfa[dfa["cidade_norm"] == top].sort_values("ts_hour")
        print(f"\nAmostra cidade (mais linhas): {top[:50]}...")
        print(s[["ts_hour", COL_PRECIP, TSF_PRED, TSF_RESID]].head(12).to_string())


compare_calc_vs_arima(DIR_D_CALC, DIR_D_ARIMA, COMPARE_YEAR, "Base D")
compare_calc_vs_arima(DIR_F_CALC, DIR_F_ARIMA, COMPARE_YEAR, "Base F")


===== Base D | 2010 =====
Linhas calculado: 2,455,408 | Linhas arima: 2,455,408
Merge outer: {'both': 9821632, 'left_only': 0, 'right_only': 0}
tsf_arima_precip_pred — NaN: 100.00% | finitos: 0.00%

Amostra cidade (mais linhas): rio pardo de minas...
                     ts_hour  PRECIPITAÇÃO TOTAL, HORÁRIO (mm)  tsf_arima_precip_pred  tsf_arima_precip_resid
1989292  2010-01-01 01:00:00                               0.0                    NaN                     NaN
1989293  2010-01-01 01:00:00                               0.0                    NaN                     NaN
1989294  2010-01-01 01:00:00                               0.0                    NaN                     NaN
1989295  2010-01-01 01:00:00                               0.0                    NaN                     NaN
1989296  2010-01-01 02:00:00                               0.0                    NaN                     NaN
1989297  2010-01-01 02:00:00                               0.0                    NaN   

## 5) Teste mínimo ARIMA em memória (uma cidade, recorte do notebook)

Reproduz a lógica `train_z` do primeiro bloco com `start=168` e tenta `ARIMA(2,1,2).fit` para ver **a exceção real** (se houver).

In [8]:
from statsmodels.tsa.arima.model import ARIMA as SM_ARIMA

TEST_YEAR = 2010
ORDER = (1, 0, 0)
H, W = 168, 720

fn = DIR_F_CALC / f"inmet_bdq_{TEST_YEAR}_cerrado.parquet"
if not fn.exists():
    print("Arquivo não encontrado:", fn)
else:
    df = pd.read_parquet(fn, columns=["cidade_norm", "ts_hour", COL_PRECIP])
    df = df.sort_values(["cidade_norm", "ts_hour"])
    df[COL_PRECIP] = pd.to_numeric(df[COL_PRECIP], errors="coerce")
    city = df["cidade_norm"].value_counts().index[0]
    sub = df[df["cidade_norm"] == city].copy()
    z = sub[COL_PRECIP].values.astype(float)
    n = len(z)
    print("Cidade:", city, "| n_horas:", n)
    print("NaN em z:", np.isnan(z).sum(), "| min/max finitos:", np.nanmin(z), np.nanmax(z))

    start = H
    train_start = max(0, start - W)
    train_z = z[train_start:start]
    print("Bloco start=", start, "len(train_z)=", len(train_z))
    print("NaN em train_z:", np.isnan(train_z).sum(), "| std:", np.nanstd(train_z))

    try:
        model = SM_ARIMA(train_z, order=ORDER, enforce_stationarity=False, enforce_invertibility=False)
        res = model.fit(method_kwargs={"maxiter": 100}, disp=False)
        fc = res.forecast(steps=min(H, n - start))
        print("OK fit+forecast. Primeiros preds:", fc[:5])
    except Exception as e:
        print("ERRO:", type(e).__name__, "—", e)

Cidade: imperatriz | n_horas: 35032
NaN em z: 4 | min/max finitos: 0.0 50.0
Bloco start= 168 len(train_z)= 168
NaN em train_z: 0 | std: 0.7197253155933646
ERRO: TypeError — ARIMA.fit() got an unexpected keyword argument 'disp'
